# ETF Valuation & Momentum Monitor
- 차트 1: 멀티플 Z-Score vs 모멘텀 스코어
- 차트 2: 멀티플 Z-Score vs 자금유입강도

각 ETF의 현재 위치와 1주일 전 위치를 화살표로 연결하여 표시

## 1. 파일 업로드

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import requests
import io
import warnings
warnings.filterwarnings('ignore')

print("필요한 파일 6개를 업로드해주세요:")
print("1. 1_pe_now.txt - 현재 기준 5년치 PER")
print("2. 2_pe_1w.txt - 1주일 전 기준 5년치 PER")
print("3. 3_return.txt - 모멘텀 스코어")
print("4. 4_flow.txt - 자금유입 데이터")
print("5. 5_pbr_now.txt - 현재 기준 5년치 PBR (BLOK-US, SOXX-US)")
print("6. 6_pbr_1w.txt - 1주일 전 기준 5년치 PBR")
print()

uploaded = files.upload()

## 2. 설정

In [ ]:
# PBR로 처리해야 하는 ETF 리스트 (추후 추가 가능)
PBR_ETF_LIST = ['BLOK-US', 'SOXX-US']

# 텔레그램 설정
BOT_TOKEN = "8328122559:AAEXkzJnxtljMON_Obt4uyb5PzJvx5-IS64"
CHAT_ID = "7481149685"

import os
os.environ['TELEGRAM_BOT_TOKEN'] = BOT_TOKEN
os.environ['TELEGRAM_CHAT_ID'] = CHAT_ID

print(f"PBR 기준 ETF: {PBR_ETF_LIST}")
print("텔레그램 설정 완료")

## 3. 데이터 로드 및 전처리

In [ ]:
def load_pe_data(filename, uploaded_files):
    """PER/PBR 데이터 로드 (컬럼=ETF, 행=날짜)"""
    content = uploaded_files[filename].decode('utf-8')
    df = pd.read_csv(io.StringIO(content), sep='\t', index_col=0)
    df.index = pd.to_datetime(df.index, format='%m/%d/%Y')
    df = df.replace('#N/A', np.nan).replace('', np.nan)
    df = df.astype(float)
    return df

def load_return_data(filename, uploaded_files):
    """모멘텀 스코어 데이터 로드 (행=ETF, 컬럼=지표)"""
    content = uploaded_files[filename].decode('utf-8')
    df = pd.read_csv(io.StringIO(content), sep='\t', index_col=0)
    df = df.replace('#N/A', np.nan).replace('', np.nan)
    # Score와 Score_1W 컬럼을 숫자로 변환
    for col in ['Score', 'Score_1W']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def load_flow_data(filename, uploaded_files):
    """자금유입 데이터 로드 (행=ETF, 컬럼=지표)"""
    content = uploaded_files[filename].decode('utf-8')
    lines = content.strip().split('\n')
    # 3번째 행부터 ETF 데이터 (0: 헤더, 1: 날짜행, 2부터: 데이터)
    header = lines[0].split('\t')
    data_lines = lines[2:]  # 3번째 행부터
    
    data = []
    for line in data_lines:
        parts = line.split('\t')
        data.append(parts)
    
    df = pd.DataFrame(data, columns=header)
    df = df.set_index(df.columns[0])
    df = df.replace('#N/A', np.nan).replace('', np.nan)
    
    # flow_Intensity 컬럼을 숫자로 변환
    for col in ['flow_Intensity', 'flow_Intensity_1W']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# 데이터 로드
print("데이터 로드 중...")
pe_now = load_pe_data('1_pe_now.txt', uploaded)
pe_1w = load_pe_data('2_pe_1w.txt', uploaded)
return_df = load_return_data('3_return.txt', uploaded)
flow_df = load_flow_data('4_flow.txt', uploaded)
pbr_now = load_pe_data('5_pbr_now.txt', uploaded)
pbr_1w = load_pe_data('6_pbr_1w.txt', uploaded)

print(f"PER Now: {pe_now.shape[0]}행 x {pe_now.shape[1]}개 ETF")
print(f"PER 1W: {pe_1w.shape[0]}행 x {pe_1w.shape[1]}개 ETF")
print(f"Return: {return_df.shape[0]}개 ETF")
print(f"Flow: {flow_df.shape[0]}개 ETF")
print(f"PBR Now: {pbr_now.shape[0]}행 x {pbr_now.shape[1]}개 ETF")
print(f"PBR 1W: {pbr_1w.shape[0]}행 x {pbr_1w.shape[1]}개 ETF")

## 4. Z-Score 계산

In [ ]:
def calculate_zscore(df, exclude_latest=True):
    """
    5년치 데이터로 Z-Score 계산
    - exclude_latest=True: 최신값 제외하고 평균/표준편차 계산 (권장)
    - Z = (현재값 - 과거평균) / 과거표준편차
    """
    result = {}
    
    for etf in df.columns:
        series = df[etf].dropna()
        
        if len(series) < 10:  # 데이터가 너무 적으면 스킵
            result[etf] = np.nan
            continue
        
        # 날짜 기준 정렬 (최신이 맨 앞)
        series = series.sort_index(ascending=False)
        
        current_value = series.iloc[0]  # 최신값
        
        if exclude_latest:
            # 최신값 제외하고 평균/표준편차 계산
            historical = series.iloc[1:]
        else:
            historical = series
        
        mean = historical.mean()
        std = historical.std()
        
        if std == 0 or np.isnan(std):
            result[etf] = np.nan
        else:
            zscore = (current_value - mean) / std
            result[etf] = zscore
    
    return result

# PER Z-Score 계산 (현재 & 1주일 전)
print("Z-Score 계산 중...")

# 현재 기준 Z-Score
zscore_now = calculate_zscore(pe_now, exclude_latest=True)
zscore_1w = calculate_zscore(pe_1w, exclude_latest=True)

# PBR 기준 ETF는 PBR Z-Score로 대체
pbr_zscore_now = calculate_zscore(pbr_now, exclude_latest=True)
pbr_zscore_1w = calculate_zscore(pbr_1w, exclude_latest=True)

for etf in PBR_ETF_LIST:
    if etf in pbr_zscore_now:
        zscore_now[etf] = pbr_zscore_now[etf]
        print(f"{etf}: PBR Z-Score 사용 (Now: {pbr_zscore_now[etf]:.2f})")
    if etf in pbr_zscore_1w:
        zscore_1w[etf] = pbr_zscore_1w[etf]
        print(f"{etf}: PBR Z-Score 사용 (1W: {pbr_zscore_1w[etf]:.2f})")

print(f"\nZ-Score 계산 완료: {len([v for v in zscore_now.values() if not np.isnan(v)])}개 ETF")

## 5. 최종 데이터 통합

In [ ]:
# 모든 ETF 리스트 (PER 데이터 기준)
all_etfs = list(pe_now.columns)

# 최종 데이터프레임 생성
final_data = []

for etf in all_etfs:
    row = {
        'ETF': etf,
        'ZScore_Now': zscore_now.get(etf, np.nan),
        'ZScore_1W': zscore_1w.get(etf, np.nan),
        'Momentum_Now': return_df.loc[etf, 'Score'] if etf in return_df.index else np.nan,
        'Momentum_1W': return_df.loc[etf, 'Score_1W'] if etf in return_df.index else np.nan,
        'Flow_Now': flow_df.loc[etf, 'flow_Intensity'] if etf in flow_df.index else np.nan,
        'Flow_1W': flow_df.loc[etf, 'flow_Intensity_1W'] if etf in flow_df.index else np.nan,
    }
    final_data.append(row)

df_final = pd.DataFrame(final_data)
df_final = df_final.set_index('ETF')

# 유효한 데이터만 필터링 (차트별로)
df_chart1 = df_final.dropna(subset=['ZScore_Now', 'ZScore_1W', 'Momentum_Now', 'Momentum_1W'])
df_chart2 = df_final.dropna(subset=['ZScore_Now', 'ZScore_1W', 'Flow_Now', 'Flow_1W'])

print(f"차트1 (멀티플 vs 모멘텀): {len(df_chart1)}개 ETF")
print(f"차트2 (멀티플 vs 플로우): {len(df_chart2)}개 ETF")

# 데이터 미리보기
print("\n=== 차트1 데이터 샘플 ===")
print(df_chart1[['ZScore_Now', 'Momentum_Now', 'ZScore_1W', 'Momentum_1W']].head(10))

## 6. 차트 생성

In [ ]:
def create_scatter_chart(df, x_col_now, x_col_1w, y_col_now, y_col_1w, 
                         title, xlabel, ylabel, filename):
    """
    스캐터 차트 생성
    - 각 ETF마다 현재(점) + 1주전(점) + 화살표 연결
    - 대각선 기준선 (좌하단 -> 우상단)
    """
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # 색상 팔레트
    colors = plt.cm.tab20(np.linspace(0, 1, len(df)))
    
    # 데이터 범위 계산 (기준선용)
    x_all = pd.concat([df[x_col_now], df[x_col_1w]])
    y_all = pd.concat([df[y_col_now], df[y_col_1w]])
    
    x_min, x_max = x_all.min(), x_all.max()
    y_min, y_max = y_all.min(), y_all.max()
    
    # 마진 추가
    x_margin = (x_max - x_min) * 0.1
    y_margin = (y_max - y_min) * 0.1
    
    # 대각선 기준선 (좌하단 -> 우상단)
    # 데이터 범위를 정규화해서 대각선 그리기
    x_range = x_max - x_min
    y_range = y_max - y_min
    
    # 기준선: 정규화된 공간에서 y=x를 원래 스케일로 변환
    # (x - x_min) / x_range = (y - y_min) / y_range
    # y = y_min + (x - x_min) * y_range / x_range
    line_x = np.array([x_min - x_margin, x_max + x_margin])
    line_y = y_min + (line_x - x_min) * y_range / x_range
    
    ax.plot(line_x, line_y, 'k--', alpha=0.5, linewidth=1.5, label='기준선')
    
    # 각 ETF 플롯
    for i, (etf, row) in enumerate(df.iterrows()):
        x_now = row[x_col_now]
        x_1w = row[x_col_1w]
        y_now = row[y_col_now]
        y_1w = row[y_col_1w]
        
        color = colors[i]
        
        # 1주전 위치 (작은 점, 연한 색)
        ax.scatter(x_1w, y_1w, c=[color], s=30, alpha=0.4, marker='o')
        
        # 현재 위치 (큰 점)
        ax.scatter(x_now, y_now, c=[color], s=80, alpha=0.9, marker='o', edgecolors='black', linewidths=0.5)
        
        # 화살표 (1주전 -> 현재)
        ax.annotate('', xy=(x_now, y_now), xytext=(x_1w, y_1w),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.6, lw=1.5))
        
        # 라벨 (현재 위치에만)
        # ETF 이름에서 -US, -HK 제거하여 간결하게
        label = etf.replace('-US', '').replace('-HK', '')
        ax.annotate(label, (x_now, y_now), fontsize=7, alpha=0.8,
                    xytext=(5, 5), textcoords='offset points')
    
    # 축 설정
    ax.set_xlim(x_min - x_margin, x_max + x_margin)
    ax.set_ylim(y_min - y_margin, y_max + y_margin)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # 그리드
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5, alpha=0.5)
    
    # 범례 (영역 설명)
    ax.text(0.02, 0.98, '← 비중 축소 고려\n(높은 밸류에이션, 낮은 모멘텀/플로우)', 
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3))
    ax.text(0.98, 0.02, '비중 확대 고려 →\n(낮은 밸류에이션, 높은 모멘텀/플로우)', 
            transform=ax.transAxes, fontsize=9, verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
    
    plt.tight_layout()
    
    # 파일 저장
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"차트 저장: {filename}")
    
    return fig

# 차트 1: 멀티플 Z-Score vs 모멘텀 스코어
fig1 = create_scatter_chart(
    df_chart1,
    x_col_now='Momentum_Now', x_col_1w='Momentum_1W',
    y_col_now='ZScore_Now', y_col_1w='ZScore_1W',
    title='ETF Valuation vs Momentum (Weekly Change)',
    xlabel='Momentum Score',
    ylabel='Valuation Z-Score (5Y PER/PBR)',
    filename='chart1_valuation_momentum.png'
)
plt.show()

# 차트 2: 멀티플 Z-Score vs 자금유입강도
fig2 = create_scatter_chart(
    df_chart2,
    x_col_now='Flow_Now', x_col_1w='Flow_1W',
    y_col_now='ZScore_Now', y_col_1w='ZScore_1W',
    title='ETF Valuation vs Flow Intensity (Weekly Change)',
    xlabel='Flow Intensity (Weekly Fund Flow / AUM)',
    ylabel='Valuation Z-Score (5Y PER/PBR)',
    filename='chart2_valuation_flow.png'
)
plt.show()

## 7. 텔레그램 전송

In [ ]:
class TelegramSender:
    def __init__(self, bot_token, chat_id):
        self.bot_token = bot_token
        self.chat_id = chat_id
        self.base_url = f"https://api.telegram.org/bot{bot_token}"
    
    def send_photo(self, photo_path, caption=None):
        """이미지 파일 전송"""
        url = f"{self.base_url}/sendPhoto"
        
        with open(photo_path, 'rb') as photo:
            files = {'photo': photo}
            data = {'chat_id': self.chat_id}
            if caption:
                data['caption'] = caption
            
            response = requests.post(url, files=files, data=data)
        
        if response.status_code == 200:
            return True
        else:
            print(f"Error: {response.text}")
            return False
    
    def send_message(self, text):
        """텍스트 메시지 전송"""
        url = f"{self.base_url}/sendMessage"
        data = {
            'chat_id': self.chat_id,
            'text': text,
            'parse_mode': 'HTML'
        }
        response = requests.post(url, data=data)
        return response.status_code == 200

# 텔레그램 전송 실행
if BOT_TOKEN and CHAT_ID:
    print("텔레그램 전송 중...")
    sender = TelegramSender(BOT_TOKEN, CHAT_ID)
    
    # 차트 1 전송
    from datetime import datetime
    date_str = datetime.now().strftime('%Y-%m-%d')
    
    if sender.send_photo('chart1_valuation_momentum.png', 
                         caption=f"📊 ETF Valuation vs Momentum ({date_str})\n"
                                 f"• 좌상단: 비중 축소 고려 (고평가 + 약모멘텀)\n"
                                 f"• 우하단: 비중 확대 고려 (저평가 + 강모멘텀)"):
        print("차트1 전송 완료")
    else:
        print("차트1 전송 실패")
    
    # 차트 2 전송
    if sender.send_photo('chart2_valuation_flow.png',
                         caption=f"📊 ETF Valuation vs Flow ({date_str})\n"
                                 f"• 좌상단: 비중 축소 고려 (고평가 + 자금유출)\n"
                                 f"• 우하단: 비중 확대 고려 (저평가 + 자금유입)"):
        print("차트2 전송 완료")
    else:
        print("차트2 전송 실패")
    
    print("\n텔레그램 전송 완료!")
else:
    print("텔레그램 설정이 필요합니다.")

## 8. 데이터 확인 (선택사항)

In [ ]:
# 전체 데이터 확인
print("=== 전체 ETF Z-Score 및 지표 ===")
display_df = df_final.copy()
display_df = display_df.round(3)
display_df = display_df.sort_values('ZScore_Now', ascending=False)
display(display_df)

In [ ]:
# 주요 ETF 위치 변화 요약
print("=== 주간 변화 요약 ===")
df_summary = df_final.copy()
df_summary['ZScore_Change'] = df_summary['ZScore_Now'] - df_summary['ZScore_1W']
df_summary['Momentum_Change'] = df_summary['Momentum_Now'] - df_summary['Momentum_1W']
df_summary['Flow_Change'] = df_summary['Flow_Now'] - df_summary['Flow_1W']

# Z-Score 상승 Top 10
print("\n[Z-Score 상승 Top 10 (밸류에이션 부담 증가)]")
print(df_summary.nlargest(10, 'ZScore_Change')[['ZScore_Now', 'ZScore_1W', 'ZScore_Change']].round(3))

# Z-Score 하락 Top 10
print("\n[Z-Score 하락 Top 10 (밸류에이션 매력 증가)]")
print(df_summary.nsmallest(10, 'ZScore_Change')[['ZScore_Now', 'ZScore_1W', 'ZScore_Change']].round(3))